In [ ]:
# import pandas as pd

# df = pd.read_csv(r"processed_outputs\all_rudder_cases_combined.csv")
# df_prop_off = pd.read_csv(r"AERODYNAMIC_DATA_propoff\propOff.csv")
import pandas as pd

df = pd.read_csv(r"CORRECTIONS_FINAL\results_propOn_FINAL\propOn_final.csv")
df_prop_off = pd.read_csv(r"CORRECTIONS_FINAL\results_propOff_FINAL\propOff_final.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Sutherland's law ---
mu_ref = 1.716e-5  # Pa·s
T_ref  = 273.15    # K
S      = 110.4     # K

def sutherland_viscosity(T):
    return mu_ref * (T / T_ref)**1.5 * (T_ref + S) / (T + S)

# --- Representative tunnel conditions (averaged from prop-on data) ---
temps = [
    290.81, 290.5, 290.63, 290.71, 290.45, 290.68, 290.57, 290.54, 290.56,
    290.77, 290.76, 290.54, 290.61, 290.53, 290.53, 290.38, 290.77, 290.42,
    290.82, 290.31, 290.83, 290.77, 290.83, 290.59, 290.69, 289.29, 289.89,
    289.33, 289.37, 289.75, 289.05, 289.33, 289.45, 289.41, 289.41, 289.39,
    289.18, 289.41, 289.48, 289.55, 289.09, 289.4,  289.11, 289.29, 289.24,
    289.39, 289.55, 289.31, 289.75, 289.81, 289.23, 289.71, 289.86, 289.23,
    289.85
]
rhos = [
    1.2111837665772662, 1.2124162911880931, 1.211909932667162,
    1.211564445727421,  1.2134526039979827, 1.2123845995887763,
    1.2128315782593753, 1.213028753275484,  1.2129332677003728,
    1.211985376693565,  1.2120390414597149, 1.2130047723183783,
    1.2127125926478157, 1.213034532873517,  1.2130705055473072,
    1.2136371492818563, 1.212021319675703,  1.2134939840084806,
    1.2117650239582876, 1.2139537833203915, 1.211735336730007,
    1.2113024600329465, 1.2110765180038419, 1.21206476484882,
    1.2116478035619338, 1.2185471305276716, 1.2160731113350067,
    1.2183907064735384, 1.2182343256648152, 1.2166486644737404,
    1.2197155774698278, 1.218474990839042,  1.2179096580800357,
    1.2180779880835713, 1.2180659507882505, 1.218162170535493,
    1.2191672579972812, 1.2181261372648537, 1.2178195442593014,
    1.2175010680190128, 1.219558861881395,  1.2181682287001427,
    1.2194624456700003, 1.2186675534126223, 1.2188902649715678,
    1.2182103230444081, 1.2175251309693753, 1.2184869752778251,
    1.2166486644737404, 1.216432841075817,  1.2187878701265824,
    1.216816646064224,  1.2162230099778601, 1.2187999149132185,
    1.2162289133388524
]

T_avg   = np.mean(temps)
rho_avg = np.mean(rhos)
mu_avg  = sutherland_viscosity(T_avg)

print(f"T_avg   = {T_avg:.3f} K")
print(f"rho_avg = {rho_avg:.6f} kg/m³")
print(f"mu_avg  = {mu_avg:.4e} Pa·s")

# --- Wing MAC from manual ---
c_mac = 0.165  # m

# --- Filter prop-off: alpha=2.5, beta=0, rudder=0, elevator=0 ---
df_re = df_prop_off[
    (df_prop_off["AoA_round"] == 2.5) &
    (df_prop_off["AoS_round"] == 0)   &
    (df_prop_off["dR"]        == 0)   &
    (df_prop_off["dE"]        == 0)
].copy()
# Keep only the first sweep (lower original row indices)
df_re = df_re[df_re.index < 8000]

# Re varies through V; rho and mu are fixed representative averages
df_re["Re"] = rho_avg * df_re["V"] * c_mac / mu_avg
df_re = df_re.sort_values("Re")
def pick_row(group):
    v = group["V_round"].iloc[0]
    if v == 30.0:
        return group.iloc[[1]]  # second occurrence
    else:
        return group.iloc[[0]]  # first occurrence

df_re = df_re.groupby("V_round", group_keys=False).apply(pick_row)
print(df_re[["V_round", "V", "Re", "CL"]].to_string())

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_re["Re"], df_re["CL"], marker="o", linestyle="-")

ax.set_xlabel(r"$Re = \rho_\mathrm{avg} \, V \, c_\mathrm{mac} \, / \, \mu_\mathrm{avg}$")
ax.set_ylabel(r"$C_L$")
ax.set_title(r"$C_L$ vs $Re$ (Prop-off, $\alpha=2.5°$, $\beta=0°$, $\delta_r=0°$, $\delta_e=0°$)")
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_images/CL_vs_Re_propoff_AoA2.5.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_filtered = df[(df["V_round"] == 40) & (df["AoS_round"] == 0) & (df["AoA_round"] == 2.5) & (df["J_round"] == 2)]
group_sorted = df_filtered.sort_values("dR")

x = -group_sorted["dR"].values
y = -group_sorted["CYaw_FINAL"].values

# Calculate linear fit
slope_cy, intercept_cy = np.polyfit(x, y, 1)
r_matrix_cy = np.corrcoef(x, y)
r_val_cy = r_matrix_cy[0, 1]
r_squared_cy = r_val_cy**2

print(f"CY Fit Function: y = {slope_cy:.4f}x + {intercept_cy:.4f}")
print(f"Correlation (r): {r_val_cy:.4f}, R^2: {r_squared_cy:.4f}\n")

# Define as function for later use
def cy_from_dr(dr_neg):
    return slope_cy * dr_neg + intercept_cy

fig, ax = plt.subplots(figsize=(8, 5))

# Plot data
ax.plot(x, y, marker="o", linestyle="None", label="Data")

# Plot fit
x_fit = np.linspace(x.min(), x.max(), 100)
y_fit = cy_from_dr(x_fit)
ax.plot(
    x_fit, y_fit, 
    linestyle="--", 
    color="blue", 
    label=rf"Fit: $y = {slope_cy:.4f}x + {intercept_cy:.4f}$ ($R^2={r_squared_cy:.4f}$)"
)

ax.set_xlabel(r"- $\delta_r$ (°)")
ax.set_ylabel(r"-$C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$")
ax.set_title(r"Rudder Deflection Angle vs $C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$ (V = 40 m/s, AoS = 0°, AoA = 2.5°, $J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2)")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.savefig("plot_images/rudder_vs_CY_V40_AoS0_AoA2.5_J2.png", dpi=150)
plt.show()
print("Saved to plot_images/rudder_vs_CY_V40_AoS0_AoA2.5_J2.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_filtered = df[(df["V_round"] == 40) & (df["AoS_round"] == 0) & (df["AoA_round"] == 2.5) & (df["J_round"] == 2)]
group_sorted = df_filtered.sort_values("dR")

x = -group_sorted["dR"].values
y = group_sorted["CMyaw"].values

# Calculate linear fit
slope_cmyaw, intercept_cmyaw = np.polyfit(x, y, 1)
r_matrix_cmyaw = np.corrcoef(x, y)
r_val_cmyaw = r_matrix_cmyaw[0, 1]
r_squared_cmyaw = r_val_cmyaw**2

print(f"CMyaw Fit Function: y = {slope_cmyaw:.4f}x + {intercept_cmyaw:.4f}")
print(f"Correlation (r): {r_val_cmyaw:.4f}, R^2: {r_squared_cmyaw:.4f}\n")

# Define as function for later use
def cmyaw_from_dr(dr_neg):
    return slope_cmyaw * dr_neg + intercept_cmyaw

fig, ax = plt.subplots(figsize=(8, 5))

# Plot data
ax.plot(x, y, marker="o", linestyle="None", label="Data")

# Plot fit
x_fit = np.linspace(x.min(), x.max(), 100)
y_fit = cmyaw_from_dr(x_fit)
ax.plot(
    x_fit, y_fit, 
    linestyle="--", 
    color="blue", 
    label=rf"Fit: $y = {slope_cmyaw:.4f}x + {intercept_cmyaw:.4f}$ ($R^2={r_squared_cmyaw:.4f}$)"
)

ax.set_xlabel(r"- $\delta_r$ (°)")
ax.set_ylabel(r"$C_{M_{yaw}} \, (2 \cdot \frac{\Pi_A}{\Pi_5 \cdot \Pi_6})$")
ax.set_title(r"Rudder Deflection Angle vs $C_{M_{yaw}} \, (2 \cdot \frac{\Pi_A}{\Pi_5 \cdot \Pi_6})$ (V = 40 m/s, AoS = 0°, AoA = 2.5°, $J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2)")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.savefig("plot_images/rudder_vs_CMyaw_V40_AoS0_AoA2.5_J2.png", dpi=150)
plt.show()
print("Saved to plot_images/rudder_vs_CMyaw_V40_AoS0_AoA2.5_J2.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_filtered = df[(df["V_round"] == 40) & (df["AoA_round"] == 2.5) & (df["J_round"] == 2)]

fig, ax = plt.subplots(figsize=(10, 6))

# Dictionary to store interpolation functions for each rudder
cmyaw_from_aos_funcs = {}

for rudder, group in df_filtered.groupby("dR"):
    group_sorted = group.sort_values("AoS_round")
    
    x = group_sorted["AoS_round"].values
    y = group_sorted["CMyaw"].values
    
    # Calculate linear fit
    slope, intercept = np.polyfit(x, y, 1)
    r_matrix = np.corrcoef(x, y)
    r_val = r_matrix[0, 1]
    r_squared = r_val**2
    
    print(f"dR = {rudder}° | Fit Function: y = {slope:.4f}x + {intercept:.4f} | r = {r_val:.4f}, R^2 = {r_squared:.4f}")
    
    # Store function in dictionary for later use
    cmyaw_from_aos_funcs[rudder] = lambda aos, m=slope, c=intercept: m * aos + c
    
    # Plot original data
    # Compatible with newer matplotlib (avoid private ax._get_lines.prop_cycler)
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
    color = colors[(len(cmyaw_from_aos_funcs) - 1) % len(colors)]
    ax.plot(
        x, y,
        marker="o",
        linestyle="None",
        color=color,
        label=fr"$\delta_r$ = {rudder}° Data",
    )
    
    # Plot linear fit
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = slope * x_fit + intercept
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={slope:.4f}x+{intercept:.4f} ($R^2={r_squared:.4f}$)",
    )

print("\n(Functions can be accessed via `cmyaw_from_aos_funcs[dR_value](aos)`)")

ax.set_xlabel(r"$\beta$ (°)")
ax.set_ylabel(r"$C_{M_{yaw}} \, (2 \cdot \frac{\Pi_A}{\Pi_5 \cdot \Pi_6})$")
ax.set_title(r"Angle of Sideslip vs $C_{M_{yaw}} \, (2 \cdot \frac{\Pi_A}{\Pi_5 \cdot \Pi_6})$ (V = 40 m/s, AoA = 2.5°, $J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2)")

# Place legend outside due to many elements
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/AoS_vs_CMyaw_V40_AoA2.5_J2_all_rudders.png", dpi=150)
plt.show()
print("Saved to plot_images/AoS_vs_CMyaw_V40_AoA2.5_J2_all_rudders.png")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Make sure output folder exists
os.makedirs("plot_images", exist_ok=True)

# -----------------------------------
# Create rounded columns if missing
# -----------------------------------
if "V_round" not in df.columns:
    df["V_round"] = df["V"].round()

if "AoS_round" not in df.columns:
    df["AoS_round"] = df["AoS"].round()

if "AoA_round" not in df.columns:
    df["AoA_round"] = df["AoA"].round(1)

if "V_round" not in df_prop_off.columns:
    df_prop_off["V_round"] = df_prop_off["V"].round()

if "AoS_round" not in df_prop_off.columns:
    df_prop_off["AoS_round"] = df_prop_off["AoS"].round()

if "AoA_round" not in df_prop_off.columns:
    df_prop_off["AoA_round"] = df_prop_off["AoA"].round(1)

# -----------------------------------
# Filter data
# -----------------------------------
df_filtered = df[
    (df["V_round"] == 40) &
    (df["J_round"] != 2) &
    (df["AoS_round"] == 0)
] .copy()

df_prop_off_filtered = df_prop_off[
    (df_prop_off["V_round"] == 40) &
    (df_prop_off["AoS_round"] == 0) &
    (df_prop_off["dR"] == 0) &
    (df_prop_off["dE"] == 0)
] .copy()

valid_groups = (
    df_filtered.groupby(["J_round", "dR"])["AoA_round"]
    .nunique()
    .reset_index()
)
valid_groups = valid_groups[valid_groups["AoA_round"] > 1]

df_prop_off_filtered = df_prop_off_filtered.sort_values("AoA_round")

# -----------------------------------
# Build CL/CD and interpolation containers
# -----------------------------------
df_filtered = df_filtered.copy()
df_prop_off_filtered = df_prop_off_filtered.copy()

df_filtered["CLCD"] = df_filtered["CL_FINAL"] / df_filtered["CD_FINAL"].replace(0, np.nan)
df_prop_off_filtered["CLCD"] = df_prop_off_filtered["CL_FINAL"] / df_prop_off_filtered["CD_FINAL"].replace(0, np.nan)

alpha_fit_max = 7.5
interpolation_functions = {
    "CL_FINAL": {},
    "CD_FINAL": {},
    "CLCD": {},
}
interpolation_coeffs = {
    "CL_FINAL": {},
    "CD_FINAL": {},
    "CLCD": {},
}

def poly_eqn_string(coeffs):
    degree = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        power = degree - i
        if power == 0:
            terms.append(f"{c:.4f}")
        elif power == 1:
            terms.append(f"{c:.4f}*x")
        else:
            terms.append(f"{c:.4f}*x^{power}")
    return " + ".join(terms).replace("+ -", "- ")

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return np.nan
    return 1.0 - ss_res / ss_tot

def fit_and_plot_vs_alpha(y_col, y_label, title, save_path, degree):
    fig, ax = plt.subplots(figsize=(11, 6))
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"] * 10)

    shown_excluded_label = False
    print(f"\n=== {y_col} interpolation (degree {degree}) ===")

    for idx, (_, row) in enumerate(valid_groups.iterrows()):
        j_val = row["J_round"]
        rudder_val = row["dR"]

        group = df_filtered[
            (df_filtered["J_round"] == j_val) &
            (df_filtered["dR"] == rudder_val)
        ].sort_values("AoA_round")

        x_all = group["AoA_round"].to_numpy(dtype=float)
        y_all = group[y_col].to_numpy(dtype=float)
        finite_mask = np.isfinite(x_all) & np.isfinite(y_all)
        x_all = x_all[finite_mask]
        y_all = y_all[finite_mask]

        used_mask = x_all <= alpha_fit_max
        x_used, y_used = x_all[used_mask], y_all[used_mask]
        x_excl, y_excl = x_all[~used_mask], y_all[~used_mask]

        color = colors[idx % len(colors)]
        base_label = fr"$J$={j_val}, $\delta_r$={rudder_val}°"

        ax.scatter(x_used, y_used, color=color, s=35, label=base_label + " used")
        if len(x_excl) > 0:
            excl_label = r"Excluded ($\alpha > 7.5^\circ$)" if not shown_excluded_label else None
            ax.scatter(
                x_excl, y_excl,
                facecolors="none",
                edgecolors=color,
                s=55,
                linewidths=1.5,
                label=excl_label,
            )
            shown_excluded_label = True

        min_points = degree + 1
        if len(x_used) < min_points:
            print(
                f"{y_col} | J={j_val}, dR={rudder_val}: skipped, "
                f"need >= {min_points} points with alpha <= {alpha_fit_max}"
            )
            continue

        coeffs = np.polyfit(x_used, y_used, degree)
        fit_func = np.poly1d(coeffs)
        y_hat = fit_func(x_used)
        r2 = compute_r2(y_used, y_hat)
        eqn_str = poly_eqn_string(coeffs)

        interpolation_coeffs[y_col][(j_val, rudder_val)] = coeffs
        interpolation_functions[y_col][(j_val, rudder_val)] = (
            lambda alpha, c=coeffs: np.polyval(c, np.asarray(alpha, dtype=float))
        )

        x_fit = np.linspace(x_used.min(), x_used.max(), 100)
        y_fit = fit_func(x_fit)
        ax.plot(
            x_fit, y_fit,
            linestyle="--",
            color=color,
            label=fr"Fit: y={eqn_str} ($R^2={r2:.4f}$)"
        )

        print(
            f"{y_col} | J={j_val}, dR={rudder_val}: "
            f"y(alpha) = {eqn_str} | R^2 = {r2:.5f}"
        )

    # Prop-off interpolation for reference
    x_off_all = df_prop_off_filtered["AoA_round"].to_numpy(dtype=float)
    y_off_all = df_prop_off_filtered[y_col].to_numpy(dtype=float)
    finite_off = np.isfinite(x_off_all) & np.isfinite(y_off_all)
    x_off_all = x_off_all[finite_off]
    y_off_all = y_off_all[finite_off]

    used_off = x_off_all <= alpha_fit_max
    x_off_used, y_off_used = x_off_all[used_off], y_off_all[used_off]
    x_off_excl, y_off_excl = x_off_all[~used_off], y_off_all[~used_off]

    ax.scatter(
        x_off_used, y_off_used,
        color="black",
        marker="s",
        s=35,
        label="Prop-off used",
    )
    if len(x_off_excl) > 0:
        excl_label = r"Prop-off excluded ($\alpha > 7.5^\circ$)"
        ax.scatter(
            x_off_excl, y_off_excl,
            facecolors="none",
            edgecolors="black",
            marker="s",
            s=60,
            linewidths=1.5,
            label=excl_label,
        )

    min_points = degree + 1
    if len(x_off_used) >= min_points:
        coeffs_off = np.polyfit(x_off_used, y_off_used, degree)
        fit_off = np.poly1d(coeffs_off)
        y_hat_off = fit_off(x_off_used)
        r2_off = compute_r2(y_off_used, y_hat_off)
        eqn_off_str = poly_eqn_string(coeffs_off)

        interpolation_coeffs[y_col][("prop_off", 0)] = coeffs_off
        interpolation_functions[y_col][("prop_off", 0)] = (
            lambda alpha, c=coeffs_off: np.polyval(c, np.asarray(alpha, dtype=float))
        )

        x_fit_off = np.linspace(x_off_used.min(), x_off_used.max(), 100)
        y_fit_off = fit_off(x_fit_off)
        ax.plot(
            x_fit_off, y_fit_off,
            linestyle="--",
            color="black",
            label=fr"Fit: y={eqn_off_str} ($R^2={r2_off:.4f}$)"
        )

        print(
            f"{y_col} | Prop-off: y(alpha) = {eqn_off_str} "
            f"| R^2 = {r2_off:.5f}"
        )
    else:
        print(
            f"{y_col} | Prop-off: skipped, need >= {min_points} points "
            f"with alpha <= {alpha_fit_max}"
        )

    ax.axvline(alpha_fit_max, color="gray", linestyle=":", linewidth=1.2)
    ax.text(
        alpha_fit_max + 0.1,
        ax.get_ylim()[0] + 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
        r"$\alpha=7.5^\circ$ cutoff",
        color="gray",
    )

    ax.set_xlabel(r"$\alpha$ (°)")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=1, loc="best")
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved to {save_path}")

# -----------------------------------
# Plot 1: CL vs alpha (linear interpolation)
# -----------------------------------
fit_and_plot_vs_alpha(
    y_col="CL_FINAL",
    y_label=r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$",
    title=r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$ vs $\alpha$ (linear fit, V = 40 m/s, AoS = 0°)",
    save_path="plot_images/CL_vs_alpha_linear_interp_upto7p5.png",
    degree=1
)

# -----------------------------------
# Plot 2: CD vs alpha (quadratic interpolation)
# -----------------------------------
fit_and_plot_vs_alpha(
    y_col="CD_FINAL",
    y_label=r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$",
    title=r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$ vs $\alpha$ (quadratic fit, V = 40 m/s, AoS = 0°)",
    save_path="plot_images/CD_vs_alpha_quadratic_interp_upto7p5.png",
    degree=2
)

# -----------------------------------
# Plot 3: CL/CD vs alpha (quadratic interpolation)
# -----------------------------------
fit_and_plot_vs_alpha(
    y_col="CLCD",
    y_label=r"$C_L / C_D$",
    title=r"$C_L / C_D$ vs $\alpha$ (quadratic fit, V = 40 m/s, AoS = 0°)",
    save_path="plot_images/CLCD_vs_alpha_quadratic_interp_upto7p5.png",
    degree=2
)

print("\nInterpolation functions are stored in `interpolation_functions`.")
print("Example usage: interpolation_functions['CL_FINAL'][(1.6, 0)](4.0)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_filtered = df[
    (df["V_round"] == 40) &
    (df["AoS_round"] == 0) &
    (df["AoA_round"] == 2.5) &
    (df["dR"].isin([0, -10]))
]

fig, ax = plt.subplots(figsize=(8, 5))

# Dictionary to store interpolation functions for each rudder
cy_from_j_funcs = {}

def outlier_mask_j(x, rudder_val):
    """Outliers are excluded from fit but still shown as hollow markers."""
    mask = np.zeros(len(x), dtype=bool)

    # Selected outlier(s) for this J vs CY plot
    if np.isclose(rudder_val, -10.0):
        mask |= np.isclose(x, 2.4)

    return mask

for rudder_val, group in df_filtered.groupby("dR"):
    group_sorted = group.sort_values("J_round")

    x = group_sorted["J_round"].to_numpy(dtype=float)
    y = group_sorted["CYaw_FINAL"].to_numpy(dtype=float)

    finite_mask = np.isfinite(x) & np.isfinite(y)
    x = x[finite_mask]
    y = y[finite_mask]
    if len(x) < 2:
        continue

    out_mask = outlier_mask_j(x, rudder_val)
    in_mask = ~out_mask

    if np.sum(in_mask) < 2:
        print(f"dR = {rudder_val}°: not enough inlier points for fit.")
        continue

    x_in, y_in = x[in_mask], y[in_mask]

    # Calculate linear fit on inliers only
    slope, intercept = np.polyfit(x_in, y_in, 1)
    r_matrix = np.corrcoef(x_in, y_in)
    r_val = r_matrix[0, 1]
    r_squared = r_val**2

    print(
        f"dR = {rudder_val}° | Fit Function: y = {slope:.4f}x + {intercept:.4f} "
        f"| r = {r_val:.4f}, R^2 = {r_squared:.4f}"
    )
    if np.any(out_mask):
        print(f"  Excluded outlier J values: {x[out_mask].tolist()}")

    # Store function in dictionary for later use
    cy_from_j_funcs[rudder_val] = lambda j, m=slope, c=intercept: m * j + c

    # Plot original data (inliers: filled)
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
    color = colors[(len(cy_from_j_funcs) - 1) % len(colors)]

    ax.scatter(
        x_in, y_in,
        marker="o",
        color=color,
        label=fr"$\delta_r$ = {rudder_val}° Data",
    )

    # Plot outliers as hollow markers
    if np.any(out_mask):
        x_out, y_out = x[out_mask], y[out_mask]
        ax.scatter(
            x_out, y_out,
            marker="o",
            facecolors="none",
            edgecolors=color,
            linewidths=1.8,
            label=fr"$\delta_r$ = {rudder_val}° Outlier",
        )

    # Plot linear fit
    x_fit = np.linspace(x_in.min(), x_in.max(), 100)
    y_fit = slope * x_fit + intercept
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={slope:.4f}x+{intercept:.4f} ($R^2={r_squared:.4f}$)",
    )

print("\n(Functions can be accessed via cy_from_j_funcs[dR_value](j))")

ax.set_xlabel(r"$J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$")
ax.set_ylabel(r"$C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$")
ax.set_title(r"$J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ vs $C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$ (V = 40 m/s, AoS = 0°, $\alpha$ = 2.5°, $\delta_r$ = 0° and -10°)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/J_vs_CY_V40_AoS0_AoA2.5_rudder0_m10_linear_fit_outliers_marked.png", dpi=150)
plt.show()
print("Saved to plot_images/J_vs_CY_V40_AoS0_AoA2.5_rudder0_m10_linear_fit_outliers_marked.png")


In [ ]:
import matplotlib.pyplot as plt

df_filtered = df[
    (df["V_round"].isin([20, 40])) &
    (df["AoS_round"] == 0) &
    (df["AoA_round"] == 2.5) &
    (df["dR"] == 0)
]

fig, ax = plt.subplots(figsize=(8, 5))

for v_val, group in df_filtered.groupby("V_round"):
    group_sorted = group.sort_values("J_round")
    ax.plot(
        group_sorted["J_round"],
        group_sorted["CFt_thrust_BEM"],
        marker="o",
        linestyle="-",
        label=f"V = {v_val} m/s",
    )

ax.set_xlabel(r"$J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$")
ax.set_ylabel(r"$C_T \, (\frac{\Pi_H}{\Pi_3^2 \cdot \Pi_4^4})$")
ax.set_title(r"$C_T \, (\frac{\Pi_H}{\Pi_3^2 \cdot \Pi_4^4})$ vs $J$ ($\delta_r$ = 0°, AoS = 0°, $\alpha$ = 2.5°)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CT_vs_J_V20_V40_AoS0_AoA2.5_rudder0.png", dpi=150)
plt.show()
print("Saved to plot_images/CT_vs_J_V20_V40_AoS0_AoA2.5_rudder0.png")


In [ ]:
import matplotlib.pyplot as plt

df_filtered = df[
    (df["V_round"] == 40) &
    (df["AoA_round"] == 2.5) &
    (df["J_round"] == 2)
]

fig, ax = plt.subplots(figsize=(8, 5))

for rudder_val, group in df_filtered.groupby("dR"):
    group_sorted = group.sort_values("AoS_round")
    ax.plot(
        group_sorted["AoS_round"],
        group_sorted["CD_FINAL"],
        marker="o",
        linestyle="-",
        label=fr"$\delta_r$ = {rudder_val}°",
    )

ax.set_xlabel(r"$\beta$ (°)")
ax.set_ylabel(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$")
ax.set_title(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$ vs $\beta$ (V = 40 m/s, $\alpha$ = 2.5°, $J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CD_vs_AoS_V40_AoA2.5_J2_all_rudders.png", dpi=150)
plt.show()
print("Saved to plot_images/CD_vs_AoS_V40_AoA2.5_J2_all_rudders.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure rounded helper columns exist
if "V_round" not in df.columns:
    df["V_round"] = df["V"].round()
if "AoS_round" not in df.columns:
    df["AoS_round"] = df["AoS"].round()
if "AoA_round" not in df.columns:
    df["AoA_round"] = df["AoA"].round(1)

if "V_round" not in df_prop_off.columns:
    df_prop_off["V_round"] = df_prop_off["V"].round()
if "AoS_round" not in df_prop_off.columns:
    df_prop_off["AoS_round"] = df_prop_off["AoS"].round()
if "AoA_round" not in df_prop_off.columns:
    df_prop_off["AoA_round"] = df_prop_off["AoA"].round(1)

# Prop-on data (same as original cell conditions)
df_on_filtered = df[
    (df["V_round"].isin([20, 40])) &
    (df["dR"].isin([0, -10])) &
    (df["J_round"] == 2) &
    (df["AoA_round"] == 2.5)
]

# Prop-off data requested: same conditions where applicable, V=40 and rudder=0
df_off_filtered = df_prop_off[
    (df_prop_off["V_round"] == 40) &
    (df_prop_off["AoA_round"] == 2.5) &
    (df_prop_off["dR"] == 0) &
    (df_prop_off["dE"] == 0)
]

def outlier_mask(x, y, case_name, v_val, rudder_val):
    """Return mask of points to exclude from fitting but still plot as hollow markers."""
    mask = np.zeros(len(x), dtype=bool)

    # User-requested outliers for prop-on
    if case_name == "prop_on" and np.isclose(v_val, 20.0) and np.isclose(rudder_val, 0.0):
        mask |= np.isclose(x, -8.0)
        mask |= np.isclose(x, 10.0)
    if case_name == "prop_on" and np.isclose(v_val, 20.0) and np.isclose(rudder_val, -10.0):
        mask |= np.isclose(x, 10.0)

    # User-requested outliers for prop-off: lowest y at beta=-10 and beta=10
    if case_name == "prop_off" and np.isclose(v_val, 40.0) and np.isclose(rudder_val, 0.0):
        for beta_target in (-10.0, 10.0):
            idx = np.where(np.isclose(x, beta_target))[0]
            if len(idx) > 0:
                lowest_idx = idx[np.argmin(y[idx])]
                mask[lowest_idx] = True

    return mask

fig, ax = plt.subplots(figsize=(10, 6))

# Store interpolation functions
cy_from_aos_funcs = {}

# Plot and fit prop-on groups
for (v_val, rudder_val), group in df_on_filtered.groupby(["V_round", "dR"]):
    group_sorted = group.sort_values("AoS_round")
    x = group_sorted["AoS_round"].to_numpy(dtype=float)
    y = group_sorted["CYaw_FINAL"].to_numpy(dtype=float)

    finite_mask = np.isfinite(x) & np.isfinite(y)
    x = x[finite_mask]
    y = y[finite_mask]
    if len(x) < 2:
        continue

    out_mask = outlier_mask(x, y, "prop_on", v_val, rudder_val)
    in_mask = ~out_mask

    if np.sum(in_mask) < 2:
        print(f"Prop-on | V={v_val}, dR={rudder_val}: not enough inlier points for fit.")
        continue

    x_in, y_in = x[in_mask], y[in_mask]

    slope, intercept = np.polyfit(x_in, y_in, 1)
    r_matrix = np.corrcoef(x_in, y_in)
    r_val = r_matrix[0, 1]
    r_squared = r_val**2

    key = ("prop_on", v_val, rudder_val)
    cy_from_aos_funcs[key] = lambda aos, m=slope, c=intercept: m * aos + c

    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0"])
    color = colors[(len(cy_from_aos_funcs) - 1) % len(colors)]

    # Inliers: filled markers
    ax.scatter(
        x_in, y_in,
        marker="o",
        color=color,
        label=fr"Prop-on V={v_val}, $\delta_r$={rudder_val}° Data",
    )

    # Outliers: hollow markers
    if np.any(out_mask):
        x_out, y_out = x[out_mask], y[out_mask]
        ax.scatter(
            x_out, y_out,
            marker="o",
            facecolors="none",
            edgecolors=color,
            linewidths=1.8,
            label=fr"Prop-on V={v_val}, $\delta_r$={rudder_val}° Outlier",
        )

    x_fit = np.linspace(x_in.min(), x_in.max(), 100)
    y_fit = slope * x_fit + intercept
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={slope:.4f}x+{intercept:.4f} ($R^2={r_squared:.4f}$)",
    )

    print(
        f"Prop-on | V={v_val}, dR={rudder_val}: "
        f"y(beta) = {slope:.4f}*beta + {intercept:.4f} | r = {r_val:.4f}, R^2 = {r_squared:.4f}"
    )
    if np.any(out_mask):
        print(f"  Excluded outlier beta values: {x[out_mask].tolist()}")

# Plot and fit prop-off (V=40, dR=0, dE=0)
group_off = df_off_filtered.sort_values("AoS_round")
x_off = group_off["AoS_round"].to_numpy(dtype=float)
y_off = group_off["CYaw_FINAL"].to_numpy(dtype=float)
finite_off = np.isfinite(x_off) & np.isfinite(y_off)
x_off = x_off[finite_off]
y_off = y_off[finite_off]

out_mask_off = outlier_mask(x_off, y_off, "prop_off", 40.0, 0.0)
in_mask_off = ~out_mask_off

if np.sum(in_mask_off) >= 2:
    x_off_in, y_off_in = x_off[in_mask_off], y_off[in_mask_off]
    slope_off, intercept_off = np.polyfit(x_off_in, y_off_in, 1)
    r_matrix_off = np.corrcoef(x_off_in, y_off_in)
    r_val_off = r_matrix_off[0, 1]
    r_squared_off = r_val_off**2

    cy_from_aos_funcs[("prop_off", 40, 0)] = lambda aos, m=slope_off, c=intercept_off: m * aos + c

    # Inliers: filled markers
    ax.scatter(
        x_off_in, y_off_in,
        marker="s",
        color="black",
        label=r"Prop-off V=40, $\delta_r$=0° Data",
    )

    # Outliers: hollow markers
    if np.any(out_mask_off):
        x_off_out, y_off_out = x_off[out_mask_off], y_off[out_mask_off]
        ax.scatter(
            x_off_out, y_off_out,
            marker="s",
            facecolors="none",
            edgecolors="black",
            linewidths=1.8,
            label=r"Prop-off V=40, $\delta_r$=0° Outlier",
        )

    x_fit_off = np.linspace(x_off_in.min(), x_off_in.max(), 100)
    y_fit_off = slope_off * x_fit_off + intercept_off
    ax.plot(
        x_fit_off, y_fit_off,
        linestyle=":",
        color="black",
        label=fr"Prop-off fit: y={slope_off:.4f}x+{intercept_off:.4f} ($R^2={r_squared_off:.4f}$)",
    )

    print(
        f"Prop-off | V=40, dR=0: "
        f"y(beta) = {slope_off:.4f}*beta + {intercept_off:.4f} | r = {r_val_off:.4f}, R^2 = {r_squared_off:.4f}"
    )
    if np.any(out_mask_off):
        print(f"  Excluded outlier beta values: {x_off[out_mask_off].tolist()}")
else:
    print("Prop-off | V=40, dR=0: not enough inlier points for linear fit.")

print("\n(Functions can be accessed via cy_from_aos_funcs[(case, V, dR)](aos))")

ax.set_xlabel(r"$\beta$ (°)")
ax.set_ylabel(r"$C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$")
ax.set_title(r"$C_Y \, (2 \cdot \frac{\Pi_I}{\Pi_6})$ vs $\beta$ ($J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2, $\alpha$ = 2.5°)")
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CY_vs_AoS_with_propOff_V40_rudder0_J2_AoA2.5_linear_fit_outliers_marked.png", dpi=150)
plt.show()
print("Saved to plot_images/CY_vs_AoS_with_propOff_V40_rudder0_J2_AoA2.5_linear_fit_outliers_marked.png")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure rounded helper columns exist
if "V_round" not in df.columns:
    df["V_round"] = df["V"].round()
if "AoS_round" not in df.columns:
    df["AoS_round"] = df["AoS"].round()
if "AoA_round" not in df.columns:
    df["AoA_round"] = df["AoA"].round(1)

if "V_round" not in df_prop_off.columns:
    df_prop_off["V_round"] = df_prop_off["V"].round()
if "AoS_round" not in df_prop_off.columns:
    df_prop_off["AoS_round"] = df_prop_off["AoS"].round()
if "AoA_round" not in df_prop_off.columns:
    df_prop_off["AoA_round"] = df_prop_off["AoA"].round(1)

# --- Conditions from original Cell 11 ---
series_specs = [
    ("prop_on", 1.6, 0.0),
    ("prop_on", 2.4, 0.0),
    ("prop_off", "off", 0.0),
]

alpha_fit_max = 7.5
degree = 1

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return np.nan
    return 1.0 - ss_res / ss_tot

def poly_eqn_string(coeffs):
    if len(coeffs) == 2:
        return f"{coeffs[0]:.4f}*x + {coeffs[1]:.4f}".replace("+ -", "- ")
    degree_local = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        p = degree_local - i
        if p == 0:
            terms.append(f"{c:.4f}")
        elif p == 1:
            terms.append(f"{c:.4f}*x")
        else:
            terms.append(f"{c:.4f}*x^{p}")
    return " + ".join(terms).replace("+ -", "- ")

cm_from_alpha_funcs = {}
cm_from_alpha_coeffs = {}

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0", "C1", "C2"] * 3)
shown_excluded_label = False

print(f"\n=== CMpitch interpolation (degree {degree}) ===")

for idx, (case_name, j_val, rudder_val) in enumerate(series_specs):
    if case_name == "prop_on":
        group = df[
            (df["V_round"] == 40) &
            (df["AoS_round"] == 0) &
            (df["dR"] == rudder_val) &
            (df["J_round"] == j_val)
        ].copy()
        base_label = fr"Prop On ($J={j_val}$, $\delta_r={rudder_val:.0f}^\circ$)"
    else:
        group = df_prop_off[
            (df_prop_off["V_round"] == 40) &
            (df_prop_off["AoS_round"] == 0) &
            (df_prop_off["dE"] == 0) &
            (df_prop_off["dR"] == rudder_val)
        ].copy()
        base_label = r"Prop Off ($delta_r=0^circ$)"
        base_label = r"Prop Off ($\delta_r=0^\circ$)"

    group = group.sort_values("AoA_round")
    x_all = group["AoA_round"].to_numpy(dtype=float)
    y_all = group["CMpitch_FINAL"].to_numpy(dtype=float)

    finite_mask = np.isfinite(x_all) & np.isfinite(y_all)
    x_all = x_all[finite_mask]
    y_all = y_all[finite_mask]
    if len(x_all) < degree + 1:
        print(f"{base_label}: skipped, insufficient valid points.")
        continue

    used_mask = x_all <= alpha_fit_max
    x_used, y_used = x_all[used_mask], y_all[used_mask]
    x_excl, y_excl = x_all[~used_mask], y_all[~used_mask]

    color = colors[idx % len(colors)]

    # Inlier/used points (filled) and excluded points (hollow), matching Cell 6 style
    ax.scatter(x_used, y_used, color=color, s=35, label=base_label + " used")
    if len(x_excl) > 0:
        excl_label = r"Excluded ($\alpha > 7.5^\circ$)" if not shown_excluded_label else None
        ax.scatter(
            x_excl, y_excl,
            facecolors="none",
            edgecolors=color,
            s=55,
            linewidths=1.5,
            label=excl_label,
        )
        shown_excluded_label = True

    if len(x_used) < degree + 1:
        print(
            f"{base_label}: skipped fit, need >= {degree + 1} points with "
            f"alpha <= {alpha_fit_max}"
        )
        continue

    coeffs = np.polyfit(x_used, y_used, degree)
    fit_fn = np.poly1d(coeffs)
    y_hat = fit_fn(x_used)
    r2 = compute_r2(y_used, y_hat)
    eqn = poly_eqn_string(coeffs)

    cm_from_alpha_coeffs[(case_name, j_val, rudder_val)] = coeffs
    cm_from_alpha_funcs[(case_name, j_val, rudder_val)] = (
        lambda alpha, c=coeffs: np.polyval(c, np.asarray(alpha, dtype=float))
    )

    x_fit = np.linspace(x_used.min(), x_used.max(), 100)
    y_fit = fit_fn(x_fit)
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={eqn} ($R^2={r2:.4f}$)",
    )

    print(f"{base_label}: y(alpha) = {eqn} | R^2 = {r2:.5f}")

ax.axvline(alpha_fit_max, color="gray", linestyle=":", linewidth=1.2)
ax.text(
    alpha_fit_max + 0.1,
    ax.get_ylim()[0] + 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
    r"$\alpha=7.5^\circ$ cutoff",
    color="gray",
)

ax.set_xlabel(r"$\alpha$ (°)")
ax.set_ylabel(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$")
ax.set_title(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$ vs $\alpha$ (linear fit, V = 40 m/s, AoS = 0°, $\delta_r$ = 0°)")
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CMpitch_vs_alpha_V40_AoS0_rudder0_linear_interp_upto7p5.png", dpi=150)
plt.show()
print("Saved to plot_images/CMpitch_vs_alpha_V40_AoS0_rudder0_linear_interp_upto7p5.png")

print("\nInterpolation functions are stored in cm_from_alpha_funcs.")
print("Example usage: cm_from_alpha_funcs[('prop_on', 1.6, 0.0)](4.0)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Sutherland's law ---
mu_ref = 1.716e-5  # Pa·s
T_ref  = 273.15    # K
S      = 110.4     # K

def sutherland_viscosity(T):
    return mu_ref * (T / T_ref)**1.5 * (T_ref + S) / (T + S)

# --- Representative tunnel conditions (averaged from prop-on data) ---
temps = [
    290.81, 290.5, 290.63, 290.71, 290.45, 290.68, 290.57, 290.54, 290.56,
    290.77, 290.76, 290.54, 290.61, 290.53, 290.53, 290.38, 290.77, 290.42,
    290.82, 290.31, 290.83, 290.77, 290.83, 290.59, 290.69, 289.29, 289.89,
    289.33, 289.37, 289.75, 289.05, 289.33, 289.45, 289.41, 289.41, 289.39,
    289.18, 289.41, 289.48, 289.55, 289.09, 289.4,  289.11, 289.29, 289.24,
    289.39, 289.55, 289.31, 289.75, 289.81, 289.23, 289.71, 289.86, 289.23,
    289.85
]
rhos = [
    1.2111837665772662, 1.2124162911880931, 1.211909932667162,
    1.211564445727421,  1.2134526039979827, 1.2123845995887763,
    1.2128315782593753, 1.213028753275484,  1.2129332677003728,
    1.211985376693565,  1.2120390414597149, 1.2130047723183783,
    1.2127125926478157, 1.213034532873517,  1.2130705055473072,
    1.2136371492818563, 1.212021319675703,  1.2134939840084806,
    1.2117650239582876, 1.2139537833203915, 1.211735336730007,
    1.2113024600329465, 1.2110765180038419, 1.21206476484882,
    1.2116478035619338, 1.2185471305276716, 1.2160731113350067,
    1.2183907064735384, 1.2182343256648152, 1.2166486644737404,
    1.2197155774698278, 1.218474990839042,  1.2179096580800357,
    1.2180779880835713, 1.2180659507882505, 1.218162170535493,
    1.2191672579972812, 1.2181261372648537, 1.2178195442593014,
    1.2175010680190128, 1.219558861881395,  1.2181682287001427,
    1.2194624456700003, 1.2186675534126223, 1.2188902649715678,
    1.2182103230444081, 1.2175251309693753, 1.2184869752778251,
    1.2166486644737404, 1.216432841075817,  1.2187878701265824,
    1.216816646064224,  1.2162230099778601, 1.2187999149132185,
    1.2162289133388524
]

T_avg   = np.mean(temps)
rho_avg = np.mean(rhos)
mu_avg  = sutherland_viscosity(T_avg)

print(f"T_avg   = {T_avg:.3f} K")
print(f"rho_avg = {rho_avg:.6f} kg/m³")
print(f"mu_avg  = {mu_avg:.4e} Pa·s")

# --- Wing MAC from manual ---
c_mac = 0.165  # m

# --- Filter prop-off: alpha=2.5, beta=0, rudder=0, elevator=0 ---
df_re = df_prop_off[
    (df_prop_off["AoA_round"] == 2.5) &
    (df_prop_off["AoS_round"] == 0)   &
    (df_prop_off["dR"]        == 0)   &
    (df_prop_off["dE"]        == 0)
].copy()


# Re varies through V; rho and mu are fixed representative averages
df_re["Re"] = rho_avg * df_re["V"] * c_mac / mu_avg
df_re = df_re.sort_values("Re")
# Keep only the lower CL point per velocity (one sweep)
df_re = df_re.loc[df_re.groupby("V_round")["CL_FINAL"].idxmin()]
df_re = df_re.sort_values("Re")

print(df_re[["V_round", "V", "Re", "CL_FINAL"]].to_string())

# --- Plot ---
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.plot(df_re["Re"], df_re["CL_FINAL"], marker="o", linestyle="-")

# ax.set_xlabel(r"$Re \, (\Pi_1^{-1}) = \rho_\mathrm{avg} \, V \, c_\mathrm{mac} \, / \, \mu_\mathrm{avg}$")
# ax.set_ylabel(r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$")
# ax.set_title(r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$ vs $Re \, (\Pi_1^{-1})$ (Prop-off, $\alpha=2.5°$, $\beta=0°$, $\delta_r=0°$, $\delta_e=0°$)")
# ax.grid(True)
# plt.tight_layout()
# plt.savefig("plot_images/CL_vs_Re_propoff_AoA2.5.png", dpi=150)
# plt.show()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_re["Re"], df_re["CL_FINAL"], marker="o", linestyle="-")
ax1.set_xlabel(r"$Re \, (\Pi_1^{-1}) = \rho_\mathrm{avg} \, V \, c_\mathrm{mac} \, / \, \mu_\mathrm{avg}$")
ax1.set_ylabel(r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$")
ax1.set_title(r"$C_L \, (2 \cdot \frac{\Pi_C}{\Pi_6})$ vs $Re \, (\Pi_1^{-1})$")
ax1.grid(True)

ax2.plot(df_re["Re"], df_re["CD_FINAL"], marker="o", linestyle="-")
ax2.set_xlabel(r"$Re \, (\Pi_1^{-1}) = \rho_\mathrm{avg} \, V \, c_\mathrm{mac} \, / \, \mu_\mathrm{avg}$")
ax2.set_ylabel(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$")
ax2.set_title(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$ vs $Re \, (\Pi_1^{-1})$")
ax2.grid(True)

fig.suptitle(r"Prop-off, $\alpha=2.5°$, $\beta=0°$, $\delta_r=0°$, $\delta_e=0°$")
plt.tight_layout()
plt.savefig("plot_images/CL_CD_vs_Re_propoff_AoA2.5.png", dpi=150)
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.makedirs("plot_images", exist_ok=True)

# Ensure rounded helper columns exist
if "V_round" not in df_prop_off.columns:
    df_prop_off["V_round"] = df_prop_off["V"].round()
if "AoS_round" not in df_prop_off.columns:
    df_prop_off["AoS_round"] = df_prop_off["AoS"].round()
if "AoA_round" not in df_prop_off.columns:
    df_prop_off["AoA_round"] = df_prop_off["AoA"].round(1)

# Base filter: Prop-off, V=40, AoS=0, dR=0
df_filtered = df_prop_off[
    (df_prop_off["V_round"] == 40) &
    (df_prop_off["AoS_round"] == 0) &
    (df_prop_off["dR"] == 0)
].copy()

alpha_fit_max = 7.5
degree = 1

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return np.nan
    return 1.0 - ss_res / ss_tot

def poly_eqn_string(coeffs):
    if len(coeffs) == 2:
        return f"{coeffs[0]:.4f}*x + {coeffs[1]:.4f}".replace("+ -", "- ")
    degree_local = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        p = degree_local - i
        if p == 0:
            terms.append(f"{c:.4f}")
        elif p == 1:
            terms.append(f"{c:.4f}*x")
        else:
            terms.append(f"{c:.4f}*x^{p}")
    return " + ".join(terms).replace("+ -", "- ")

# Dictionaries to store interpolation functions and coefficients per delta_e
cm_from_alpha_funcs_de = {}
cm_from_alpha_coeffs_de = {}

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0", "C1", "C2", "C3", "C4", "C5"] * 2)
shown_excluded_label = False

print(f"\n=== CMpitch interpolation per delta_e (degree {degree}, alpha <= {alpha_fit_max}°) ===")

# Get unique elevator deflections sorted
unique_de = sorted(df_filtered["dE"].unique())

for idx, de in enumerate(unique_de):
    group = df_filtered[df_filtered["dE"] == de].copy()
    group = group.sort_values("AoA_round")
    
    x_all = group["AoA_round"].to_numpy(dtype=float)
    y_all = group["CMpitch_FINAL"].to_numpy(dtype=float)
    
    finite_mask = np.isfinite(x_all) & np.isfinite(y_all)
    x_all = x_all[finite_mask]
    y_all = y_all[finite_mask]
    
    if len(x_all) == 0:
        print(f"dE = {de:>5}°: No valid data.")
        continue
    
    base_label = fr"$\delta_e$ = {de}°"
    
    # Split into used (within alpha_fit_max) and excluded (beyond alpha_fit_max)
    used_mask = x_all <= alpha_fit_max
    x_used, y_used = x_all[used_mask], y_all[used_mask]
    x_excl, y_excl = x_all[~used_mask], y_all[~used_mask]
    
    color = colors[idx % len(colors)]
    
    # Plot used points (filled) and excluded points (hollow)
    ax.scatter(x_used, y_used, color=color, s=35, label=base_label + " used")
    if len(x_excl) > 0:
        excl_label = r"Excluded ($\alpha > 7.5^\circ$)" if not shown_excluded_label else None
        ax.scatter(
            x_excl, y_excl,
            facecolors="none",
            edgecolors=color,
            s=55,
            linewidths=1.5,
            label=excl_label,
        )
        shown_excluded_label = True
    
    if len(x_used) < degree + 1:
        print(
            f"dE = {de:>5}°: skipped fit, need >= {degree + 1} points with "
            f"alpha <= {alpha_fit_max}°. Available: {len(x_used)}"
        )
        continue
    
    # Fit using only used points
    coeffs = np.polyfit(x_used, y_used, degree)
    fit_fn = np.poly1d(coeffs)
    y_hat = fit_fn(x_used)
    r2 = compute_r2(y_used, y_hat)
    eqn = poly_eqn_string(coeffs)
    
    # Store functions and coefficients keyed by delta_e
    cm_from_alpha_coeffs_de[de] = coeffs
    cm_from_alpha_funcs_de[de] = (
        lambda alpha, c=coeffs: np.polyval(c, np.asarray(alpha, dtype=float))
    )
    
    # Plot fit line
    x_fit = np.linspace(x_used.min(), x_used.max(), 100)
    y_fit = fit_fn(x_fit)
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={eqn} ($R^2={r2:.4f}$)",
    )
    
    print(f"dE = {de:>5}°: y(alpha) = {eqn} | R^2 = {r2:.5f} | n_used={len(x_used)}, n_excl={len(x_excl)}")

ax.axvline(alpha_fit_max, color="gray", linestyle=":", linewidth=1.2)
ax.text(
    alpha_fit_max + 0.1,
    ax.get_ylim()[0] + 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
    r"$\alpha=7.5^\circ$ cutoff",
    color="gray",
)

ax.set_xlabel(r"$\alpha$ (°)")
ax.set_ylabel(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$")
ax.set_title(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$ vs $\alpha$ per $\delta_e$ (linear fit, Prop-off, V = 40 m/s, AoS = 0°)")
ax.legend(fontsize=8, loc='best')
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CMpitch_vs_alpha_propoff_V40_AoS0_per_elevator_linear_interp_upto7p5.png", dpi=150)
plt.show()

print("\nInterpolation functions stored in cm_from_alpha_funcs_de, keyed by delta_e.")
print("Example usage: cm_from_alpha_funcs_de[-10](4.0) for delta_e=-10°")


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("plot_images", exist_ok=True)

# Ensure rounded helper columns exist
if "V_round" not in df_prop_off.columns:
    df_prop_off["V_round"] = df_prop_off["V"].round()
if "AoS_round" not in df_prop_off.columns:
    df_prop_off["AoS_round"] = df_prop_off["AoS"].round()
if "AoA_round" not in df_prop_off.columns:
    df_prop_off["AoA_round"] = df_prop_off["AoA"].round(1)

# Base filter: Prop-off, V=40, AoS=0, dR=0
df_cd_filtered = df_prop_off[
    (df_prop_off["V_round"] == 40) &
    (df_prop_off["AoS_round"] == 0) &
    (df_prop_off["dR"] == 0)
].copy()

alpha_fit_max = 7.5
degree = 2

def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return np.nan
    return 1.0 - ss_res / ss_tot

def poly_eqn_string(coeffs):
    degree_local = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        p = degree_local - i
        if p == 0:
            terms.append(f"{c:.4f}")
        elif p == 1:
            terms.append(f"{c:.4f}*x")
        else:
            terms.append(f"{c:.4f}*x^{p}")
    return " + ".join(terms).replace("+ -", "- ")

# Dictionaries to store interpolation functions and coefficients per delta_e
cd_from_alpha_funcs_de = {}
cd_from_alpha_coeffs_de = {}

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0", "C1", "C2", "C3", "C4", "C5"] * 2)
shown_excluded_label = False

print(f"\n=== CD interpolation per delta_e (degree {degree}, alpha <= {alpha_fit_max} deg) ===")

# Get unique elevator deflections sorted
unique_de = sorted(df_cd_filtered["dE"].unique())

for idx, de in enumerate(unique_de):
    group = df_cd_filtered[df_cd_filtered["dE"] == de].copy()
    group = group.sort_values("AoA_round")

    x_all = group["AoA_round"].to_numpy(dtype=float)
    y_all = group["CD_FINAL"].to_numpy(dtype=float)

    finite_mask = np.isfinite(x_all) & np.isfinite(y_all)
    x_all = x_all[finite_mask]
    y_all = y_all[finite_mask]

    if len(x_all) == 0:
        print(f"dE = {de:>5} deg: No valid data.")
        continue

    base_label = fr"$\delta_e$ = {de} deg"

    # Split into used (within alpha_fit_max) and excluded (beyond alpha_fit_max)
    used_mask = x_all <= alpha_fit_max
    x_used, y_used = x_all[used_mask], y_all[used_mask]
    x_excl, y_excl = x_all[~used_mask], y_all[~used_mask]

    color = colors[idx % len(colors)]

    # Plot used points (filled) and excluded points (hollow)
    ax.scatter(x_used, y_used, color=color, s=35, label=base_label + " used")
    if len(x_excl) > 0:
        excl_label = r"Excluded ($\alpha > 7.5^\circ$)" if not shown_excluded_label else None
        ax.scatter(
            x_excl, y_excl,
            facecolors="none",
            edgecolors=color,
            s=55,
            linewidths=1.5,
            label=excl_label,
        )
        shown_excluded_label = True

    if len(x_used) < degree + 1:
        print(
            f"dE = {de:>5} deg: skipped fit, need >= {degree + 1} points with "
            f"alpha <= {alpha_fit_max} deg. Available: {len(x_used)}"
        )
        continue

    # Quadratic fit using only used points
    coeffs = np.polyfit(x_used, y_used, degree)
    fit_fn = np.poly1d(coeffs)
    y_hat = fit_fn(x_used)
    r2 = compute_r2(y_used, y_hat)
    eqn = poly_eqn_string(coeffs)

    # Store functions and coefficients keyed by delta_e
    cd_from_alpha_coeffs_de[de] = coeffs
    cd_from_alpha_funcs_de[de] = (
        lambda alpha, c=coeffs: np.polyval(c, np.asarray(alpha, dtype=float))
    )

    # Plot fit line
    x_fit = np.linspace(x_used.min(), x_used.max(), 120)
    y_fit = fit_fn(x_fit)
    ax.plot(
        x_fit, y_fit,
        linestyle="--",
        color=color,
        label=fr"Fit: y={eqn} ($R^2={r2:.4f}$)",
    )

    print(f"dE = {de:>5} deg: y(alpha) = {eqn} | R^2 = {r2:.5f} | n_used={len(x_used)}, n_excl={len(x_excl)}")

ax.axvline(alpha_fit_max, color="gray", linestyle=":", linewidth=1.2)
ax.text(
    alpha_fit_max + 0.1,
    ax.get_ylim()[0] + 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
    r"$\alpha=7.5^\circ$ cutoff",
    color="gray",
)

ax.set_xlabel(r"$\alpha$ (deg)")
ax.set_ylabel(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$")
ax.set_title(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$ vs $\alpha$ per $\delta_e$ (quadratic fit, Prop-off, V = 40 m/s, AoS = 0 deg)")
ax.legend(fontsize=8, loc="best")
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CD_vs_alpha_propoff_V40_AoS0_per_elevator_quadratic_interp_upto7p5.png", dpi=150)
plt.show()

print("\nInterpolation functions stored in cd_from_alpha_funcs_de, keyed by delta_e.")
print("Example usage: cd_from_alpha_funcs_de[-10](4.0) for delta_e=-10 deg")

In [ ]:
import matplotlib.pyplot as plt

def select_best_sweep(group, gap_threshold=100):
    if group is None or group.empty:
        return group
    
    group = group.sort_index().copy()
    row_jump = group.index.to_series().diff().fillna(0)
    group["sweep_id"] = (row_jump > gap_threshold).cumsum()

    best_block = None
    best_range = -1
    best_npoints = -1

    for _, block in group.groupby("sweep_id"):
        aoa_range = block["AoA"].max() - block["AoA"].min()
        npoints = len(block)

        if (aoa_range > best_range) or (aoa_range == best_range and npoints > best_npoints):
            best_block = block.copy()
            best_range = aoa_range
            best_npoints = npoints

    return best_block

# Filter prop on J=1.6, dR=0
df_on_filtered_16_r0 = df[
    (df["V_round"] == 40) & 
    (df["AoS_round"] == 0) & 
    (df["dR"] == 0) & 
    (df["J_round"] == 1.6)
]
df_on_filtered_16_r0 = select_best_sweep(df_on_filtered_16_r0)
if df_on_filtered_16_r0 is not None: df_on_filtered_16_r0 = df_on_filtered_16_r0.sort_values("AoA_round")

# Filter prop on J=2.4, dR=0
df_on_filtered_24_r0 = df[
    (df["V_round"] == 40) & 
    (df["AoS_round"] == 0) & 
    (df["dR"] == 0) & 
    (df["J_round"] == 2.4)
]
df_on_filtered_24_r0 = select_best_sweep(df_on_filtered_24_r0)
if df_on_filtered_24_r0 is not None: df_on_filtered_24_r0 = df_on_filtered_24_r0.sort_values("AoA_round")

# Filter prop on J=1.6, dR=-10
df_on_filtered_16_rm10 = df[
    (df["V_round"] == 40) & 
    (df["AoS_round"] == 0) & 
    (df["dR"] == -10) & 
    (df["J_round"] == 1.6)
]
df_on_filtered_16_rm10 = select_best_sweep(df_on_filtered_16_rm10)
if df_on_filtered_16_rm10 is not None: df_on_filtered_16_rm10 = df_on_filtered_16_rm10.sort_values("AoA_round")

# Filter prop on J=2.4, dR=-10
df_on_filtered_24_rm10 = df[
    (df["V_round"] == 40) & 
    (df["AoS_round"] == 0) & 
    (df["dR"] == -10) & 
    (df["J_round"] == 2.4)
]
df_on_filtered_24_rm10 = select_best_sweep(df_on_filtered_24_rm10)
if df_on_filtered_24_rm10 is not None: df_on_filtered_24_rm10 = df_on_filtered_24_rm10.sort_values("AoA_round")

# Filter prop off, dR=0
df_off_filtered = df_prop_off[
    (df_prop_off["V_round"] == 40) & 
    (df_prop_off["AoS_round"] == 0) & 
    (df_prop_off["dE"] == 0) &
    (df_prop_off["dR"] == 0)
]
df_off_filtered = select_best_sweep(df_off_filtered)
if df_off_filtered is not None: df_off_filtered = df_off_filtered.sort_values("AoA_round")

fig, ax = plt.subplots(figsize=(10, 6))

if df_on_filtered_16_r0 is not None and not df_on_filtered_16_r0.empty:
    ax.plot(
        df_on_filtered_16_r0["AoA_round"],
        df_on_filtered_16_r0["CMpitch_FINAL"],
        marker="o",
        linestyle="-",
        color="C0",
        label=r"Prop On ($J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 1.6, $\delta_r$ = 0°)"
    )

if df_on_filtered_24_r0 is not None and not df_on_filtered_24_r0.empty:
    ax.plot(
        df_on_filtered_24_r0["AoA_round"],
        df_on_filtered_24_r0["CMpitch_FINAL"],
        marker="^",
        linestyle="-",
        color="C1",
        label=r"Prop On ($J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2.4, $\delta_r$ = 0°)"
    )

if df_on_filtered_16_rm10 is not None and not df_on_filtered_16_rm10.empty:
    ax.plot(
        df_on_filtered_16_rm10["AoA_round"],
        df_on_filtered_16_rm10["CMpitch_FINAL"],
        marker="o",
        linestyle="--",
        color="C0",
        label=r"Prop On ($J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 1.6, $\delta_r$ = -10°)"
    )

if df_on_filtered_24_rm10 is not None and not df_on_filtered_24_rm10.empty:
    ax.plot(
        df_on_filtered_24_rm10["AoA_round"],
        df_on_filtered_24_rm10["CMpitch_FINAL"],
        marker="^",
        linestyle="--",
        color="C1",
        label=r"Prop On ($J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ = 2.4, $\delta_r$ = -10°)"
    )

if df_off_filtered is not None and not df_off_filtered.empty:
    ax.plot(
        df_off_filtered["AoA_round"],
        df_off_filtered["CMpitch_FINAL"],
        marker="s",
        linestyle=":",
        color="gray",
        label=r"Prop Off ($\delta_r$ = 0°)"
    )

ax.set_xlabel(r"$\alpha$ (°)")
ax.set_ylabel(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$")
ax.set_title(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$ vs $\alpha$ (V = 40 m/s, AoS = 0°)")
ax.legend(fontsize=9, ncol=2)
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CMpitch_vs_alpha_V40_AoS0_rudder0_m10.png", dpi=150)
plt.show()
print("Saved to plot_images/CMpitch_vs_alpha_V40_AoS0_rudder0_m10.png")

In [ ]:
import matplotlib.pyplot as plt

# Filter prop on
df_on_j = df[
    (df["V_round"] == 40) & 
    (df["AoA_round"] == 2.5) &
    (df["AoS_round"] == 0) &
    (df["dR"] == 0)
].sort_values("J_round")

# Filter prop off and average
df_off_j = df_prop_off[
    (df_prop_off["V_round"] == 40) & 
    (df_prop_off["AoA_round"] == 2.5) &
    (df_prop_off["AoS_round"] == 0) &
    (df_prop_off["dR"] == 0) &
    (df_prop_off["dE"] == 0)
]

cmpitch_off_avg = df_off_j["CMpitch_FINAL"].mean() if not df_off_j.empty else None

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    df_on_j["J_round"],
    df_on_j["CMpitch_FINAL"],
    marker="o",
    linestyle="-",
    label="Prop On"
)

if cmpitch_off_avg is not None:
    ax.axhline(
        y=cmpitch_off_avg, 
        color='r', 
        linestyle=':', 
        label=f"Prop Off (avg: {cmpitch_off_avg:.4f})"
    )

ax.set_xlabel(r"$J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$")
ax.set_ylabel(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$")
ax.set_title(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$ vs $J \, (\frac{1}{\Pi_3 \cdot \Pi_4})$ (V = 40 m/s, $\alpha$ = 2.5°, $\beta$ = 0°, $\delta_r$ = 0°)")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig("plot_images/CMpitch_vs_J_V40_AoS0_AoA2.5_rudder0.png", dpi=150)
plt.show()
print("Saved to plot_images/CMpitch_vs_J_V40_AoS0_AoA2.5_rudder0.png")

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
from scipy.optimize import brentq

# Target values
cl_target = 0.77
j_val = 2.4
dR = 0

# Get the interpolation coefficients for J=2.4, dR=0
key = (j_val, dR)
if key in interpolation_coeffs['CL_FINAL']:
    coeffs = interpolation_coeffs['CL_FINAL'][key]
    print(f"Found interpolation coefficients for J={j_val}, dR={dR}°")
    print(f"Coefficients: {coeffs}")
    
    # Define the CL function
    cl_func = interpolation_functions['CL_FINAL'][key]
    
    # For linear fit: CL = a*alpha + b
    # So: alpha = (CL - b) / a
    if len(coeffs) == 2:  # Linear fit
        a, b = coeffs
        alpha_result = (cl_target - b) / a if not np.isclose(a, 0) else np.nan
        print(f"\nLinear interpolation: CL = {a:.6f}*alpha + {b:.6f}")
        print(f"\nFor CL = {cl_target}:")
        print(f"alpha = ({cl_target} - {b:.6f}) / {a:.6f}")
        print(f"alpha = {alpha_result:.4f}°")
    else:
        # For higher-order polynomials, use numerical root finding
        def equation(alpha):
            return cl_func(alpha) - cl_target
        
        # Find the root of (CL(alpha) - CL_target) = 0
        try:
            alpha_result = brentq(equation, -15, 15)
            print(f"\nFor CL = {cl_target}:")
            print(f"alpha = {alpha_result:.4f}°")
            print(f"Verification: CL({alpha_result:.4f}°) = {cl_func(alpha_result):.6f}")
        except ValueError as e:
            print(f"Error finding root: {e}")
else:
    print(f"Interpolation data not found for J={j_val}, dR={dR}°")
    print(f"Available keys: {list(interpolation_coeffs['CL_FINAL'].keys())}")

In [ ]:
# Use the alpha value to find Cmpitch from J=2.4 interpolation
alpha_to_use = alpha_result  # 5.0434°
j_val_cm = 2.4
dR_cm = 0.0

# Access the Cmpitch interpolation function for J=2.4, dR=0
key_cm = ('prop_on', j_val_cm, dR_cm)
if key_cm in cm_from_alpha_funcs:
    cmpitch_result = cm_from_alpha_funcs[key_cm](alpha_to_use)
    coeffs_cm = cm_from_alpha_coeffs[key_cm]
    
    print(f"Using alpha = {alpha_to_use:.4f}° (from CL = 0.77)")
    print(f"\nCMpitch interpolation for J={j_val_cm}, δr={dR_cm}°:")
    print(f"Coefficients: {coeffs_cm}")
    
    # For linear: CMpitch = a*alpha + b
    if len(coeffs_cm) == 2:
        a_cm, b_cm = coeffs_cm
        print(f"CMpitch = {a_cm:.6f}*alpha + {b_cm:.6f}")
        print(f"\nFor alpha = {alpha_to_use:.4f}°:")
        print(f"CMpitch = {a_cm:.6f}*{alpha_to_use:.4f} + {b_cm:.6f}")
        print(f"CMpitch = {cmpitch_result:.6f}")
    else:
        print(f"\nFor alpha = {alpha_to_use:.4f}°:")
        print(f"CMpitch = {cmpitch_result:.6f}")
else:
    print(f"CMpitch interpolation not found for key {key_cm}")
    print(f"Available keys: {list(cm_from_alpha_funcs.keys())}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Inputs from previous steps
trim_alpha = float(alpha_result)        # alpha from CL=0.77 at J=1.6, dR=0
target_cmpitch = -float(cmpitch_result)  # target Cmpitch from J=1.6 interpolation

# 1) Evaluate all prop-off Cmpitch(alpha) functions at trim alpha
if len(cm_from_alpha_funcs_de) == 0:
    raise RuntimeError("cm_from_alpha_funcs_de is empty. Run the prop-off Cmpitch-vs-alpha interpolation cell first.")

delta_e_values = []
cmpitch_values = []

for de in sorted(cm_from_alpha_funcs_de.keys()):
    cm_val = float(cm_from_alpha_funcs_de[de](trim_alpha))
    delta_e_values.append(float(de))
    cmpitch_values.append(cm_val)

x_de = np.asarray(delta_e_values, dtype=float)
y_cm = np.asarray(cmpitch_values, dtype=float)

print(f"Trim alpha used: {trim_alpha:.4f} deg")
print("\nCmpitch at trim alpha for each elevator deflection (prop-off):")
for de, cm_val in zip(x_de, y_cm):
    print(f"  dE={de:>6.1f} deg -> Cmpitch={cm_val:+.6f}")

# 2) Build interpolation function dE -> Cmpitch and plot
# Use linear interpolation/fit for trim relation
degree_de_cm = 1
coeffs_de_cm = np.polyfit(x_de, y_cm, degree_de_cm)
fit_de_to_cm = np.poly1d(coeffs_de_cm)

# R^2 of dE -> Cmpitch fit
y_hat = fit_de_to_cm(x_de)
ss_res = np.sum((y_cm - y_hat) ** 2)
ss_tot = np.sum((y_cm - np.mean(y_cm)) ** 2)
r2_de_cm = np.nan if np.isclose(ss_tot, 0.0) else 1.0 - ss_res / ss_tot

a_de_cm, b_de_cm = coeffs_de_cm
print(f"\nInterpolation (dE -> Cmpitch): Cmpitch = {a_de_cm:.6f}*dE + {b_de_cm:.6f} | R^2={r2_de_cm:.5f}")

# Plot
x_plot = np.linspace(x_de.min(), x_de.max(), 300)
y_plot = fit_de_to_cm(x_plot)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_de, y_cm, s=45, label=fr"Data at alpha={trim_alpha:.3f} deg")
ax.plot(x_plot, y_plot, "--", linewidth=1.8, label=fr"Fit: Cmpitch={a_de_cm:.4f} dE + {b_de_cm:.4f}")

ax.axhline(target_cmpitch, color="black", linestyle=":", linewidth=1.2, label=fr"Target Cmpitch={target_cmpitch:.4f}")
ax.set_xlabel(r"$\delta_e$ (deg)")
ax.set_ylabel(r"$C_{M_{pitch}} \, (2 \cdot \frac{\Pi_B}{\Pi_6})$")
ax.set_title(fr"$\delta_e$ vs $C_{{M_{{pitch}}}}$ at trim alpha={trim_alpha:.3f} deg (prop-off)")
ax.grid(True)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_images/delta_e_vs_CMpitch_at_trim_alpha_propoff.png", dpi=150)
plt.show()
print("Saved to plot_images/delta_e_vs_CMpitch_at_trim_alpha_propoff.png")

# 3) Invert interpolation to find trim dE for target Cmpitch
def f_root(de):
    return fit_de_to_cm(de) - target_cmpitch

de_min, de_max = float(x_de.min()), float(x_de.max())
f_min, f_max = f_root(de_min), f_root(de_max)

if f_min == 0:
    delta_e_trim = de_min
elif f_max == 0:
    delta_e_trim = de_max
elif f_min * f_max < 0:
    # Root bracketed inside data range
    delta_e_trim = brentq(f_root, de_min, de_max)
else:
    # No bracket in range -> use linear analytic inversion (extrapolation if needed)
    if np.isclose(a_de_cm, 0.0):
        delta_e_trim = np.nan
    else:
        delta_e_trim = (target_cmpitch - b_de_cm) / a_de_cm

cmpitch_check = float(fit_de_to_cm(delta_e_trim)) if np.isfinite(delta_e_trim) else np.nan

print(f"\nTarget Cmpitch to counter: {target_cmpitch:+.6f}")
print(f"Trim elevator deflection from interpolation: dE_trim = {delta_e_trim:+.4f} deg")
print(f"Check: Cmpitch(dE_trim) = {cmpitch_check:+.6f}")

if delta_e_trim < de_min or delta_e_trim > de_max:
    print(f"Warning: dE_trim is outside interpolation data range [{de_min:.2f}, {de_max:.2f}] deg (extrapolated).")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Uses alpha_result from the CL target cell and cd_from_alpha_funcs_de from the CD-vs-alpha interpolation cell
if "alpha_result" not in globals():
    raise RuntimeError("alpha_result is not defined. Run the cell that computes alpha_result first.")
if "cd_from_alpha_funcs_de" not in globals() or len(cd_from_alpha_funcs_de) == 0:
    raise RuntimeError("cd_from_alpha_funcs_de is empty. Run the CD-vs-alpha-per-elevator interpolation cell first.")

alpha_for_cd = float(alpha_result)

de_values = []
cd_values = []
for de in sorted(cd_from_alpha_funcs_de.keys()):
    cd_val = float(cd_from_alpha_funcs_de[de](alpha_for_cd))
    de_values.append(float(de))
    cd_values.append(cd_val)

x_de = np.asarray(de_values, dtype=float)
y_cd = np.asarray(cd_values, dtype=float)

print(f"alpha_result used: {alpha_for_cd:.4f} deg")
print("\nCD at alpha_result for each elevator deflection:")
for de, cd in zip(x_de, y_cd):
    print(f"  dE={de:>6.1f} deg -> CD={cd:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_de, y_cd, marker="o", linestyle="-", label=fr"$C_D$ at $\alpha={alpha_for_cd:.3f}^\circ$")
ax.set_xlabel(r"$\delta_e$ (deg)")
ax.set_ylabel(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$")
ax.set_title(fr"$\delta_e$ vs $C_D$ at $\alpha={alpha_for_cd:.3f}^\circ$ (prop-off)")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.savefig("plot_images/delta_e_vs_CD_at_alpha_result_propoff.png", dpi=150)
plt.show()
print("Saved to plot_images/delta_e_vs_CD_at_alpha_result_propoff.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Requires alpha_result, delta_e_trim and cd_from_alpha_funcs_de from earlier cells
if "alpha_result" not in globals():
    raise RuntimeError("alpha_result is not defined. Run the CL target/interpolation cell first.")
if "delta_e_trim" not in globals() or not np.isfinite(delta_e_trim):
    raise RuntimeError("delta_e_trim is not available or invalid. Run the trim elevator cell first.")
if "cd_from_alpha_funcs_de" not in globals() or len(cd_from_alpha_funcs_de) == 0:
    raise RuntimeError("cd_from_alpha_funcs_de is empty. Run the CD-vs-alpha-per-elevator interpolation cell first.")

alpha_for_cd = float(alpha_result)
de_trim = float(delta_e_trim)
de_ref = 0.0

# Build CD values at alpha_result for each elevator deflection
de_vals = np.asarray(sorted(cd_from_alpha_funcs_de.keys()), dtype=float)
cd_vals = np.asarray([float(cd_from_alpha_funcs_de[de](alpha_for_cd)) for de in de_vals], dtype=float)

# Quadratic interpolation/fit: CD = a*dE^2 + b*dE + c
degree = 2
coeffs_cd_de = np.polyfit(de_vals, cd_vals, degree)
fit_cd_from_de = np.poly1d(coeffs_cd_de)
cd_at_de_trim = float(fit_cd_from_de(de_trim))
cd_at_de0 = float(fit_cd_from_de(de_ref))
delta_cd_due_elevator = cd_at_de_trim - cd_at_de0

# Goodness of fit
cd_hat = fit_cd_from_de(de_vals)
ss_res = np.sum((cd_vals - cd_hat) ** 2)
ss_tot = np.sum((cd_vals - np.mean(cd_vals)) ** 2)
r2_cd_de = np.nan if np.isclose(ss_tot, 0.0) else 1.0 - ss_res / ss_tot

a_cd, b_cd, c_cd = coeffs_cd_de
print(f"alpha_result used: {alpha_for_cd:.4f} deg")
print(f"Quadratic fit: CD(dE) = {a_cd:.8f}*dE^2 + {b_cd:.8f}*dE + {c_cd:.8f} | R^2={r2_cd_de:.6f}")
print(f"Input trim elevator: dE_trim = {de_trim:+.4f} deg")
print(f"Predicted CD at dE_trim: CD = {cd_at_de_trim:.6f}")
print(f"Predicted CD at dE = 0 deg: CD = {cd_at_de0:.6f}")
print(f"Delta CD due to elevator deflection (trim - zero): {delta_cd_due_elevator:+.6f}")

# Plot
x_fit = np.linspace(de_vals.min(), de_vals.max(), 300)
y_fit = fit_cd_from_de(x_fit)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(de_vals, cd_vals, s=45, label=fr"Data at $\alpha={alpha_for_cd:.3f}^\circ$")
ax.plot(x_fit, y_fit, "--", linewidth=1.8, label=fr"Quadratic fit ($R^2={r2_cd_de:.4f}$)")
ax.scatter([de_trim], [cd_at_de_trim], marker="x", s=90, linewidths=2.2, color="red", label=fr"$dE_{{trim}}={de_trim:.2f}^\circ$ -> $CD={cd_at_de_trim:.5f}$")
ax.scatter([de_ref], [cd_at_de0], marker="D", s=50, color="black", label=fr"$dE=0^\circ$ -> $CD={cd_at_de0:.5f}$")

ax.set_xlabel(r"$\delta_e$ (deg)")
ax.set_ylabel(r"$C_D \, (2 \cdot \frac{\Pi_D}{\Pi_6})$")
ax.set_title(fr"Quadratic $\delta_e$-$C_D$ interpolation at $\alpha={alpha_for_cd:.3f}^\circ$ (prop-off)")
ax.grid(True)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_images/delta_e_vs_CD_quadratic_at_alpha_result_with_trim_point.png", dpi=150)
plt.show()
print("Saved to plot_images/delta_e_vs_CD_quadratic_at_alpha_result_with_trim_point.png")